# Reinforcement Fine-Tuning on tau-bench Retail

This notebook demonstrates **Reinforcement Fine-Tuning (RFT)** using Azure OpenAI's o4-mini model on the tau-bench retail dataset.

## Why RFT instead of Supervised Fine-Tuning?

| Approach | How it learns | Best for |
|----------|---------------|----------|
| **SFT** | "Here's the correct answer, learn it" | Tasks with single correct outputs |
| **RFT** | "Here's a score, improve yourself" | Tasks with multiple valid paths |

For tool selection tasks, there's rarely one "correct" answer — the order might vary, multiple tools might work, context matters. RFT lets the model **explore strategies** and learn from rewards rather than memorizing fixed patterns.

## What's in this notebook?

1. **Data preparation**: Load tau-bench tasks (prepared in notebook 01)
2. **Grader validation**: Test the scoring function locally
3. **Training**: Launch RFT job on Azure OpenAI
4. **Monitoring**: Track job progression

## Dataset: tau-bench retail

[tau-bench](https://github.com/sierra-research/tau-bench) is a benchmark for evaluating LLM agents on realistic tasks. The retail domain includes 115 customer service scenarios with ground-truth tool sequences.

## Environment Setup

In [1]:
# Install dependencies (uncomment if needed)
# !pip install -r ../requirements.txt --quiet

## Configuration

Load centralized configuration from `src/settings.py`.

In [2]:
import sys
sys.path.insert(0, '..')  # Add parent to path for src imports

from src.settings import (
    DATA_DIR, DATASETS_DIR, CONFIG_DIR, OUTPUTS_DIR,
    AZURE_DEPLOYMENT, print_config,
    load_system_prompt, load_tool_definitions
)

print_config()

📁 Root dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..
📁 Data dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\data
📁 Config dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\src\config
📁 Graders dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\src\graders
📁 Outputs dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\outputs
🔗 Endpoint: https://test-finetuning-eastus2-eaf.cognitiveservices.azure.com/
📦 Baseline deployment: o4-mini
🆕 Vanilla deployment: gpt-5.2
🎯 Fine-tuned deployment: planner-low-0103-0303-step30


## Client Initialization

Initialize the OpenAI client pointing to Azure OpenAI endpoint.

In [3]:
from src.azure_client import get_client, test_connection

client = get_client()
test_connection(client)

✅ Connection OK - o4-mini


True

## Training Configuration

Toggle between quick test mode (fewer samples, faster iteration) and full training mode.

| Mode | Train | Validation | Use case |
|------|-------|------------|----------|
| Quick test | 15 | 10 | Validate grader logic, debug format issues |
| Full training | 85 | 20 | Complete training run with all available data |

In [4]:
# =============================================================================
# CONFIGURATION - Modify this for your training run
# =============================================================================
QUICK_TEST = False  # Set to False for full training
STRUCTURED_OUTPUT = False  # Enable JSON structured output during training

if QUICK_TEST:
    NUM_TRAIN, NUM_VAL = 15, 10
    print("⚡ QUICK TEST MODE")
else:
    NUM_TRAIN, NUM_VAL = 85, 20
    print("🚀 FULL TRAINING MODE")

print(f"Train: {NUM_TRAIN}, Val: {NUM_VAL}, Total needed: {NUM_TRAIN + NUM_VAL}")
print(f"Structured output: {'ENABLED' if STRUCTURED_OUTPUT else 'DISABLED'}")

🚀 FULL TRAINING MODE
Train: 85, Val: 20, Total needed: 105
Structured output: DISABLED


## Load Data

Load the prepared train/val/test splits from notebook 01.

In [5]:
from src.data_utils import load_train_val_test, print_data_stats

train_samples, val_samples, test_samples = load_train_val_test(DATASETS_DIR)
print_data_stats(train_samples, val_samples, test_samples)

# Load system prompt and tool definitions
SYSTEM_PROMPT = load_system_prompt()
TOOL_DEFINITIONS = load_tool_definitions()

print(f"📝 System prompt: {len(SYSTEM_PROMPT)} chars")
print(f"🔧 Tools: {len(TOOL_DEFINITIONS)}")

📦 Data loaded:
   Train: 85 samples
   Val:   20 samples
   Test:  10 samples
   Total: 115 samples
📝 System prompt: 1325 chars
🔧 Tools: 15


---

## Grader Design

The grader is **the most critical component** of RFT. It defines what "good" means — the model optimizes entirely based on this signal.

### Our Metrics

| Metric | Description |
|--------|-------------|
| **Recall** | Are the expected tools mentioned? |
| **Precision** | Ratio of correct tools to predicted tools |
| **F2 Score** | Recall-weighted F-score (β=2) |

```python
F2 = 5 * (precision * recall) / (4 * precision + recall)
```

### Why F2?

F2 score prioritizes recall over precision, which is appropriate for retail customer service:
- **Low recall** (missing tools) → Workflow fails completely → F2 drops significantly
- **Low precision** (extra tools) → Minor API cost → F2 drops less

In customer service, missing a required tool causes the entire workflow to fail. An extra tool just adds a small API call cost. F2 (β=2) weights recall 2x more important than precision.

### Scoring Examples

| Scenario | Recall | Precision | F2 |
|----------|--------|-----------|-----|
| Perfect match | 1.0 | 1.0 | **1.0** |
| Missing 1 tool (3/4) | 0.75 | 1.0 | **0.79** |
| +1 extra tool (4/5) | 1.0 | 0.80 | **0.95** |
| +2 extra tools (4/6) | 1.0 | 0.67 | **0.91** |
| Half correct (2/4) | 0.50 | 1.0 | **0.56** |

### Test the Grader Locally

**Always test your grader before training.** A buggy grader = wasted compute + wrong learned behavior.

In [6]:
from src.graders import grade, get_grader_config
from src.graders.tests import run_all_tests

# Run comprehensive tests
run_all_tests()

GRADER TESTS (F2 Score) - Text Parsing

Test 1 - Perfect response: 1.000 (expected 1.0) [PASS]
Test 2 - Missing 1 tool (3/4): 0.789 (expected 0.789) [PASS]
Test 3 - All tools (any order): 1.000 (expected 1.0) [PASS]
Test 4 - One extra tool: 0.922 (expected 0.922) [PASS]
Test 5 - Empty response: 0.000 (expected 0.0) [PASS]
Test 6 - Wrong tools: 0.000 (expected 0.0) [PASS]
Test 7 - Two extra tools: 0.849 (expected 0.849) [PASS]
Test 8 - No expected tools, no prediction: 1.000 (expected 1.0) [PASS]
Test 9 - No expected tools, prediction made: 0.000 (expected 0.0) [PASS]
Test 10 - Half tools correct: 0.556 (expected 0.556) [PASS]

All grader tests passed!

------------------------------------------------------------
GRADER TESTS (F2 Score) - JSON Structured Output
------------------------------------------------------------

JSON Test 1 - JSON - Perfect response: 1.000 (expected 1.0) [PASS]
JSON Test 2 - JSON - Missing 1 tool: 0.789 (expected 0.789) [PASS]
JSON Test 3 - JSON - One extra to

True

### Grader Configuration

Get the grader configuration for the training job.

In [7]:
GRADER_CONFIG = get_grader_config()
print(f"✅ Grader type: {GRADER_CONFIG['type']}")
print(f"✅ Grader name: {GRADER_CONFIG['name']}")
print(f"✅ Code length: {len(GRADER_CONFIG['source'])} chars")

✅ Grader type: python
✅ Grader name: planner_grader
✅ Code length: 4533 chars


---

## Upload to Azure OpenAI

Upload training and validation files. Files must be processed before they can be used in a fine-tuning job.

In [8]:
from src.training import wait_for_file

TRAIN_FILE = DATASETS_DIR / "train.jsonl"
VAL_FILE = DATASETS_DIR / "val.jsonl"

# Upload files
with open(TRAIN_FILE, "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with open(VAL_FILE, "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")

print(f"✅ Train: {train_file.id}")
print(f"✅ Val: {val_file.id}")

✅ Train: file-c7c9813ffb83484a8926f0a90af48bc3
✅ Val: file-946163b263764a1aa3005050d781c12d


In [9]:
# Wait for files to be processed
print("⏳ Waiting for files to be processed...")
wait_for_file(client, train_file.id)
wait_for_file(client, val_file.id)
print("✅ Files ready")

⏳ Waiting for files to be processed...
  file-c7c9813ffb83484a8926f0a90af48bc3: processed
  file-946163b263764a1aa3005050d781c12d: pending
  file-946163b263764a1aa3005050d781c12d: processed
✅ Files ready


---

## Create RFT Job

Launch the reinforcement fine-tuning job with our grader configuration.

### RFT Hyperparameters

| Parameter | Range | Default | Description |
|-----------|-------|---------|-------------|
| `n_epochs` | integer | auto | Number of passes through training data |
| `batch_size` | integer | auto | Training batch size |
| `learning_rate_multiplier` | 0.02-0.2 | auto | Learning rate multiplier |
| `reasoning_effort` | low/medium/high | medium | Model reasoning effort during training |
| `compute_multiplier` | 0.5-3.0 | auto | Exploration compute (low=underfit, high=overfit) |
| `eval_interval` | 1-25 | auto (~1) | Training steps between evaluations |
| `eval_samples` | 1-10 | auto (~5) | Samples per evaluation |

### Structured Output (response_format)

When `STRUCTURED_OUTPUT = True`, the model is trained to produce JSON with this schema:

```json
{
  "reasoning": "Step-by-step explanation of tool selection",
  "tools": ["tool1", "tool2", "tool3"]
}
```

This enables reliable parsing and eliminates text extraction heuristics. The grader supports both JSON and text fallback for backward compatibility.

> **Tip**: Start with defaults (`"auto"`), then adjust based on training metrics.

In [10]:
from datetime import datetime
from src.training import save_job_history
from src.settings import load_planner_schema

# =============================================================================
# HYPERPARAMETERS - Modify for your training run
# Use "auto" to let Azure choose optimal values, or set specific values
# =============================================================================
N_EPOCHS = 3                        # "auto" or integer
BATCH_SIZE = "auto"                 # "auto" or integer
LEARNING_RATE_MULTIPLIER = "auto"   # "auto" or 0.02-0.2
REASONING_EFFORT = "low"            # "low", "medium", "high"
COMPUTE_MULTIPLIER = "auto"         # "auto" or 0.5-3.0
EVAL_INTERVAL = "auto"              # "auto" or 1-25
EVAL_SAMPLES = "auto"               # "auto" or 1-10
# =============================================================================

# Build hyperparameters dict (exclude "auto" values - Azure will use defaults)
hyperparams = {}
if N_EPOCHS != "auto":
    hyperparams["n_epochs"] = N_EPOCHS
if BATCH_SIZE != "auto":
    hyperparams["batch_size"] = BATCH_SIZE
if LEARNING_RATE_MULTIPLIER != "auto":
    hyperparams["learning_rate_multiplier"] = LEARNING_RATE_MULTIPLIER
if REASONING_EFFORT:  # Always set reasoning_effort
    hyperparams["reasoning_effort"] = REASONING_EFFORT
if COMPUTE_MULTIPLIER != "auto":
    hyperparams["compute_multiplier"] = COMPUTE_MULTIPLIER
if EVAL_INTERVAL != "auto":
    hyperparams["eval_interval"] = EVAL_INTERVAL
if EVAL_SAMPLES != "auto":
    hyperparams["eval_samples"] = EVAL_SAMPLES

print(f"Hyperparameters: {hyperparams}")

# Load response_format schema if structured output is enabled
RESPONSE_FORMAT = load_planner_schema() if STRUCTURED_OUTPUT else None
if RESPONSE_FORMAT:
    print(f"📋 Structured output: ENABLED ({RESPONSE_FORMAT['json_schema']['name']})")
else:
    print("📋 Structured output: DISABLED (text mode)")

# Include reasoning effort in suffix for easy identification
run_suffix = f"planner-{REASONING_EFFORT}-{datetime.now().strftime('%m%d-%H%M')}"

# Build reinforcement config
reinforcement_config = {
    "grader": GRADER_CONFIG,
    "hyperparameters": hyperparams
}

# Add response_format if structured output is enabled
if RESPONSE_FORMAT is not None:
    reinforcement_config["response_format"] = RESPONSE_FORMAT

job = client.fine_tuning.jobs.create(
    model="o4-mini-2025-04-16",
    training_file=train_file.id,
    validation_file=val_file.id,
    method={
        "type": "reinforcement",
        "reinforcement": reinforcement_config
    },
    suffix=run_suffix
)

JOB_ID = job.id
print(f"Job: {JOB_ID}")
print(f"Suffix: {run_suffix}")

# Save job history with all configured hyperparameters
save_job_history(
    job_id=JOB_ID,
    suffix=run_suffix,
    train_file_id=train_file.id,
    val_file_id=val_file.id,
    train_samples=len(train_samples),
    val_samples=len(val_samples),
    test_samples=len(test_samples),
    hyperparameters=hyperparams
)

Hyperparameters: {'n_epochs': 3, 'reasoning_effort': 'low'}
📋 Structured output: DISABLED (text mode)
Job: ftjob-fdcd885213864da095b3c8cbe9450018
Suffix: planner-low-0128-1740
💾 Saved to c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\outputs\job_history.json


WindowsPath('c:/Projects/Agentic-Reinforcement-Fine-Tuning/notebooks/../outputs/job_history.json')

---

## Monitor Training

Monitor the job until completion. Press Ctrl+C to stop monitoring (job continues on Azure).

In [11]:
from src.training import monitor_job

try:
    final_job = monitor_job(client, JOB_ID)
except KeyboardInterrupt:
    print(f"\n⚠️ Monitoring stopped. Job continues on Azure: {JOB_ID}")

📊 Monitoring ftjob-fdcd885213864da095b3c8cbe9450018
   Press Ctrl+C to stop (job continues on Azure)

[0m] pending
.....
[6m] running
......................................................................................................................................
[142m] succeeded

🏁 succeeded
   Model: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740


### Reload after kernel restart (optional)

If you restarted the kernel, run this cell to reload the job info.

In [12]:
# Uncomment to reload after kernel restart
'''
import sys
sys.path.insert(0, '..')

from src.azure_client import get_client
from src.training import load_job_history

client = get_client()
job_info = load_job_history()
JOB_ID = job_info["job_id"]

print(f"✅ Reloaded JOB_ID: {JOB_ID}")
'''

'\nimport sys\nsys.path.insert(0, \'..\')\n\nfrom src.azure_client import get_client\nfrom src.training import load_job_history\n\nclient = get_client()\njob_info = load_job_history()\nJOB_ID = job_info["job_id"]\n\nprint(f"✅ Reloaded JOB_ID: {JOB_ID}")\n'

---

## Finalize

After training completes successfully, update the job history and .env file.

In [16]:
from src.training.job_utils import finalize_successful_job

# Check status and finalize if successful
job = client.fine_tuning.jobs.retrieve(JOB_ID)
print(f"Status: {job.status}")

if job.status == "succeeded":
    finalize_successful_job(client, JOB_ID)

Status: succeeded
💾 Updated c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\outputs\job_history.json
✅ Updated .env: FINETUNED_DEPLOYMENT=planner-low-0128-1740
✅ Model: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740

🎉 Run 02_eval.ipynb to evaluate!


---

## Next Steps

🎉 **Training complete!**

Continue to **`03_deployment.ipynb`** to:
- Deploy the fine-tuned model
- Compare baseline vs fine-tuned performance
- Visualize the improvements